* 1.匯入讀取資料
* 2.基礎資料分析
* 3.基礎資料處理
* 4.資料分割(二分)
* 5.載入資料及與切分
* 6.訓練多個模型(各種都試試看)
* 網址: https://www.kaggle.com/datasets/scientificstephen/medical-examination-dataset-analysis

## 🧩 步驟 1：匯入資料

In [ ]:
import pandas as pd
df = pd.read_csv("archive/medical_examination.csv")
print("資料維度:", df.shape)
# 查看前幾筆資料
df.head()


## 🧩 步驟 2：基礎資料分析

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# 設定字型為支援中文的字型（以繁體中文常用字型為例）
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 微軟正黑體
plt.rcParams['axes.unicode_minus'] = False  # 讓負號正常顯示


# 檢查資料結構與缺值
print(df.info())
print(df.isnull().sum())

# 加入 age_years 欄位（將 age 從天轉換成年）
df['age_years'] = (df['age'] / 365).astype(int)

# 移除不合理的血壓值
df = df[(df['ap_hi'] > df['ap_lo']) & (df['ap_hi'] > 0) & (df['ap_lo'] > 0)]

# 去除 gender 中的錯誤值
df = df[df['gender'].isin([1, 2])]

# 使用 IQR（四分位距）法去除離群值
cols = ['height', 'weight', 'ap_hi', 'ap_lo']

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("去除離群值後資料筆數：", df.shape)

# 去除重複值
df = df.drop_duplicates()
# ---------- 移除 'age' 和 'id' 欄位 ----------
if 'age' in df.columns:
    df = df.drop(columns=['age'])
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# cardio 分佈
print(df['cardio'].value_counts(normalize=True))

# 年齡分佈圖
sns.histplot(df['age_years'], bins=30)
plt.title("年齡分佈")
plt.show()

# 皮爾森相關性熱圖
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("皮爾森相關係數熱圖")
plt.show()


#### 目標(標籤)欄位分布

In [ ]:
# ---------- 目標欄位分布 ----------
import seaborn as sns
import matplotlib.pyplot as plt
# 設定字型為支援中文的字型（以繁體中文常用字型為例）
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 微軟正黑體
plt.rcParams['axes.unicode_minus'] = False  # 讓負號正常顯示

sns.countplot(x='cardio', data=df)
plt.title("心血管疾病分布 (0=否, 1=是)")
plt.xlabel("是否被診斷為心血管疾病")
plt.ylabel("人數")
plt.show()

##### ---------- 觀察重要特徵 ----------
* 年齡（age_years，以天為單位）
* 性別（gender）
* 身高(height)、體重(weight)、血壓（ap_hi, ap_lo）
* 血糖(gluc)、膽固醇(cholesterol)、吸菸(smoke)、酒精(alco)、運動(active)
* cardio 是標籤（0 = 無病，1 = 有心血管疾病）

In [ ]:
# ---------- 特徵欄位分類 ----------
continuous_features = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']
categorical_features = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# 1. age 轉成年已於上一步完成
# 2. 去除收縮壓 ap_hi 小於舒張壓 ap_lo 或不合理（負值）已於上一步完成
# 3. 去除 gender 中的錯誤值已於上一步完成
# 4. 去除離群值已於上一步完成
# 5. 去除重複值上一步完成

# ---------- 繪製連續變數與 cardio 的關係 ----------
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.boxplot(
        x='cardio', y=col, data=df, ax=axes[i],
        showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "black"},
        palette="coolwarm"
    )
    axes[i].set_title(f"{col} 與心血管疾病關係", fontsize=13)
    axes[i].set_xlabel("心血管疾病 (0=否, 1=是)", fontsize=11)
    axes[i].set_ylabel(col, fontsize=11)

# 隱藏多餘子圖格
for j in range(len(continuous_features), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# ---------- 繪製分類變數與 cardio 的關係 ----------
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    sns.countplot(x=col, hue='cardio', data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} 與心血管疾病關係", fontsize=13)
    axes[i].set_xlabel(col, fontsize=11)
    axes[i].set_ylabel("數量", fontsize=11)
    axes[i].legend(title="心血管疾病", labels=["否(0)", "是(1)"])

plt.tight_layout()
plt.show()

## 🧩 步驟 3：基礎資料處理

In [ ]:
# 1. age 轉成年已於上一步完成
# 2. 去除收縮壓 ap_hi 小於舒張壓 ap_lo 或不合理（負值）已於上一步完成
# 3. 去除 gender 中的錯誤值已於上一步完成
# 4. 去除上下 1% 極端值已於上一步完成
# 5. 去除重複值上一步完成

# 6. 刪除無用欄位
X = df.drop(columns=['cardio'])
y = df['cardio']


print("清理後資料筆數：", df.shape[0])
print("資料維度:", df.shape)

In [ ]:
# ----------------------------------------------------
# 匯入套件
# ----------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

# ----------------------------------------------------
# 資料（已前處理完畢）
# ----------------------------------------------------
# df 為清理後的資料集
X = df.drop(columns=['cardio'])
y = df['cardio']

# 數值與類別欄位
num_cols = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# ----------------------------------------------------
# 建立不同的前處理器
# ----------------------------------------------------
# 標準化 + OneHot
pre_std_onehot = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# 正規化 + OneHot
pre_minmax_onehot = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# 標準化（無 OneHot）
pre_std = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
])

# 正規化（無 OneHot）
pre_minmax = ColumnTransformer([
    ('num', MinMaxScaler(), num_cols)
])

# ----------------------------------------------------
# 超參數搜尋空間
# ----------------------------------------------------
param_grid = {
    'classifier__n_neighbors': list(range(1, 21)),
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2]
}

## 🧩 步驟 4、5：資料分割、訓練多個模型

### ✅ KNN 模型訓練與評估

In [ ]:
# ----------------------------------------------------
# Step 1: 分出獨立測試集
# ----------------------------------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ----------------------------------------------------
# Step 2: 四種資料處理策略
# ----------------------------------------------------
preprocessors = {
    "正規化+OneHot": pre_minmax_onehot,
    "標準化+OneHot": pre_std_onehot,
    "標準化": pre_std,
    "正規化": pre_minmax
}

# ----------------------------------------------------
# Step 3: 超參數設定
# ----------------------------------------------------
param_grid = {
    'classifier__n_neighbors': list(range(1, 21)),
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2]
}

results = []

# ----------------------------------------------------
# Step 4: Train/Val/Test 四組
# ----------------------------------------------------
for name, preprocessor in preprocessors.items():
    print(f"\n===== Train/Val/Test ({name}) =====")
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', KNeighborsClassifier())
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    best_params = grid.best_params_
    val_acc = grid.best_score_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": "Train/Val/Test",
        "資料處理方式": name,
        "最佳參數": best_params,
        "驗證平均準確率 (CV)": val_acc,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 5: k-Fold (5, 10)
# ----------------------------------------------------
for k in [5, 10]:
    print(f"\n===== Train/Val/Test ({k}-Fold) =====")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    pipe = Pipeline([
        ('preprocessor', pre_std_onehot),
        ('classifier', KNeighborsClassifier())
    ])
    grid = GridSearchCV(pipe, param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    best_params = grid.best_params_
    val_acc = grid.best_score_

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": f"Train/Val/Test ({k}-Fold)",
        "資料處理方式": "標準化+OneHot",
        "最佳參數": best_params,
        "驗證平均準確率 (CV)": val_acc,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 6: 統一結果表格 + 標出最佳組合
# ----------------------------------------------------
df_results = pd.DataFrame(results)

best_acc_idx = df_results["測試 Accuracy"].idxmax()
best_f1_idx = df_results["F1"].idxmax()
df_results.loc[best_acc_idx, "🏆最佳Accuracy"] = "⬅️"
df_results.loc[best_f1_idx, "🏆最佳F1"] = "⬅️"

pd.set_option('display.max_colwidth', None)
display(df_results.sort_values(by="測試 Accuracy", ascending=False).reset_index(drop=True))
best_knn_model = grid.best_estimator_

#### KNN 各資料處理方式之 AUC 比較

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 統一風格（和 Decision Tree 圖一樣）
sns.set(style="whitegrid", font="Microsoft JhengHei", font_scale=1.1)
palette = sns.color_palette("Set2")

# ---------- 圖1：Accuracy ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_results, x="資料處理方式", y="測試 Accuracy", hue="資料切分", palette=palette)
plt.title("KNN 各資料處理方式之測試集 Accuracy 比較")
plt.ylabel("Accuracy")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.65, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖2：F1 Score ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_results, x="資料處理方式", y="F1", hue="資料切分", palette=palette)
plt.title("KNN 各資料處理方式之 F1 Score 比較")
plt.ylabel("F1 Score")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.6, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖3：AUC ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_results, x="資料處理方式", y="AUC", hue="資料切分", palette=palette)
plt.title("KNN 各資料處理方式之 AUC 比較")
plt.ylabel("ROC AUC")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.7, 0.85)
plt.tight_layout()
plt.show()


### ✅ Decision Tree 模型

In [ ]:
# ----------------------------------------------------
# Step 1: 分出獨立測試集
# ----------------------------------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ----------------------------------------------------
# Step 2: 資料處理方式
# ----------------------------------------------------
preprocessors = {
    "正規化": pre_minmax,
    "正規化+OneHot": pre_minmax_onehot,
    "標準化": pre_std,
    "標準化+OneHot": pre_std_onehot
}

# ----------------------------------------------------
# Step 3: 調參範圍 (適合 Decision Tree)
# ----------------------------------------------------
param_grid = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [3, 5, 7, 9, 11, None],
    'classifier__min_samples_split': [2, 5, 10, 20],
    'classifier__min_samples_leaf': [1, 2, 4]
}

results = []

# ----------------------------------------------------
# Step 4: Train/Test (5-Fold) - 四種資料處理
# ----------------------------------------------------
for name, preprocessor in preprocessors.items():
    print(f"\n===== Train/Test (5-Fold) - {name} =====")
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(random_state=42))
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": "Train/Test (5-Fold)",
        "資料處理方式": name,
        "最佳參數": grid.best_params_,
        "驗證平均準確率 (CV)": grid.best_score_,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 5: Train/Test (10-Fold)
# ----------------------------------------------------
for k in [10]:
    print(f"\n===== Train/Test ({k}-Fold) - 標準化+OneHot =====")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    pipe = Pipeline([
        ('preprocessor', pre_std_onehot),
        ('classifier', DecisionTreeClassifier(random_state=42))
    ])
    grid = GridSearchCV(pipe, param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": f"Train/Test ({k}-Fold)",
        "資料處理方式": "標準化+OneHot",
        "最佳參數": grid.best_params_,
        "驗證平均準確率 (CV)": grid.best_score_,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 6: 統一表格輸出
# ----------------------------------------------------
df_dt = pd.DataFrame(results)

best_acc_idx = df_dt["測試 Accuracy"].idxmax()
best_f1_idx = df_dt["F1"].idxmax()
df_dt.loc[best_acc_idx, "🏆最佳Accuracy"] = "⬅️"
df_dt.loc[best_f1_idx, "🏆最佳F1"] = "⬅️"

pd.set_option('display.max_colwidth', None)
display(df_dt.sort_values(by="測試 Accuracy", ascending=False).reset_index(drop=True))
best_dt_model = grid.best_estimator_

#### Decision Tree 各資料處理方式之 AUC 比較

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 設定圖表風格
sns.set(style="whitegrid", font="Microsoft JhengHei", font_scale=1.1)

# 建立一個顏色列表（讓每組圖顏色一致）
palette = sns.color_palette("Set2")

# ---------- 圖1：Accuracy ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_dt, x="資料處理方式", y="測試 Accuracy", hue="資料切分", palette=palette)
plt.title("Decision Tree 各資料處理方式之測試集 Accuracy 比較")
plt.ylabel("Accuracy")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.65, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖2：F1 Score ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_dt, x="資料處理方式", y="F1", hue="資料切分", palette=palette)
plt.title("Decision Tree 各資料處理方式之 F1 Score 比較")
plt.ylabel("F1 Score")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.6, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖3：AUC ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_dt, x="資料處理方式", y="AUC", hue="資料切分", palette=palette)
plt.title("Decision Tree 各資料處理方式之 AUC 比較")
plt.ylabel("ROC AUC")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.7, 0.85)
plt.tight_layout()
plt.show()


### ✅ Random Forest 模型

In [ ]:
# ----------------------------------------------------
# Step 1: 分出獨立測試集
# ----------------------------------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ----------------------------------------------------
# Step 2: 四種資料處理方式
# ----------------------------------------------------
preprocessors = {
    "正規化": pre_minmax,
    "正規化+OneHot": pre_minmax_onehot,
    "標準化": pre_std,
    "標準化+OneHot": pre_std_onehot
}

# ----------------------------------------------------
# Step 3: 超參數範圍 (適合 Random Forest)
# ----------------------------------------------------
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__criterion': ['gini', 'entropy']
}

results = []

# ----------------------------------------------------
# Step 4: Train/Test (5-Fold) - 四種資料處理
# ----------------------------------------------------
for name, preprocessor in preprocessors.items():
    print(f"\n===== Train/Test (5-Fold) - {name} =====")
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": "Train/Test (5-Fold)",
        "資料處理方式": name,
        "最佳參數": grid.best_params_,
        "驗證平均準確率 (CV)": grid.best_score_,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 5: Train/Test (10-Fold)
# ----------------------------------------------------
for k in [10]:
    print(f"\n===== Train/Test ({k}-Fold) - 標準化+OneHot =====")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    pipe = Pipeline([
        ('preprocessor', pre_std_onehot),
        ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
    ])
    grid = GridSearchCV(pipe, param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": f"Train/Test ({k}-Fold)",
        "資料處理方式": "標準化+OneHot",
        "最佳參數": grid.best_params_,
        "驗證平均準確率 (CV)": grid.best_score_,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 6: 統一結果表格 + 標註最佳組合
# ----------------------------------------------------
df_rf = pd.DataFrame(results)

best_acc_idx = df_rf["測試 Accuracy"].idxmax()
best_f1_idx = df_rf["F1"].idxmax()
df_rf.loc[best_acc_idx, "🏆最佳Accuracy"] = "⬅️"
df_rf.loc[best_f1_idx, "🏆最佳F1"] = "⬅️"

pd.set_option('display.max_colwidth', None)
display(df_rf.sort_values(by="測試 Accuracy", ascending=False).reset_index(drop=True))
best_rf_model = grid.best_estimator_

#### Random Forest 各資料處理方式之 AUC 比較

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 設定統一圖表樣式
sns.set(style="whitegrid", font="Microsoft JhengHei", font_scale=1.1)
palette = sns.color_palette("Set2")

# ---------- 圖1：Accuracy ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_rf, x="資料處理方式", y="測試 Accuracy", hue="資料切分", palette=palette)
plt.title("Random Forest 各資料處理方式之測試集 Accuracy 比較")
plt.ylabel("Accuracy")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.65, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖2：F1 Score ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_rf, x="資料處理方式", y="F1", hue="資料切分", palette=palette)
plt.title("Random Forest 各資料處理方式之 F1 Score 比較")
plt.ylabel("F1 Score")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.6, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖3：AUC ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_rf, x="資料處理方式", y="AUC", hue="資料切分", palette=palette)
plt.title("Random Forest 各資料處理方式之 AUC 比較")
plt.ylabel("ROC AUC")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.7, 0.85)
plt.tight_layout()
plt.show()


### ✅ Naive Bayes 模型

In [ ]:
# ----------------------------------------------------
# Step 1: 分出獨立測試集
# ----------------------------------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ----------------------------------------------------
# Step 2: 四種資料處理方式
# ----------------------------------------------------
preprocessors = {
    "正規化": pre_minmax,
    "正規化+OneHot": pre_minmax_onehot,
    "標準化": pre_std,
    "標準化+OneHot": pre_std_onehot
}

# ----------------------------------------------------
# Step 3: Naive Bayes 可調參數
# ----------------------------------------------------
param_grid = [
    {'classifier': [GaussianNB()]},
    {'classifier': [BernoulliNB()], 'classifier__alpha': [0.1, 0.5, 1.0]},
    {'classifier': [MultinomialNB()], 'classifier__alpha': [0.1, 0.5, 1.0]}
]

results = []

# ----------------------------------------------------
# Step 4: Train/Test (5-Fold) - 四種資料處理
# ----------------------------------------------------
for name, preprocessor in preprocessors.items():
    print(f"\n===== Train/Test (5-Fold) - {name} =====")
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', GaussianNB())
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": "Train/Test (5-Fold)",
        "資料處理方式": name,
        "最佳參數": grid.best_params_,
        "驗證平均準確率 (CV)": grid.best_score_,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 5: Train/Test (10-Fold)
# ----------------------------------------------------
for k in [10]:
    print(f"\n===== Train/Test ({k}-Fold) - 標準化+OneHot =====")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    pipe = Pipeline([
        ('preprocessor', pre_std_onehot),
        ('classifier', GaussianNB())
    ])
    grid = GridSearchCV(pipe, param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_full, y_train_full)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "資料切分": f"Train/Test ({k}-Fold)",
        "資料處理方式": "標準化+OneHot",
        "最佳參數": grid.best_params_,
        "驗證平均準確率 (CV)": grid.best_score_,
        "測試 Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

# ----------------------------------------------------
# Step 6: 統一結果表格 + 標註最佳組合
# ----------------------------------------------------
df_nb = pd.DataFrame(results)

best_acc_idx = df_nb["測試 Accuracy"].idxmax()
best_f1_idx = df_nb["F1"].idxmax()
df_nb.loc[best_acc_idx, "🏆最佳Accuracy"] = "⬅️"
df_nb.loc[best_f1_idx, "🏆最佳F1"] = "⬅️"

pd.set_option('display.max_colwidth', None)
display(df_nb.sort_values(by="測試 Accuracy", ascending=False).reset_index(drop=True))
best_nb_model = grid.best_estimator_

#### Naive Bayes 各資料處理方式之 AUC 比較

In [ ]:
sns.set(style="whitegrid", font="Microsoft JhengHei", font_scale=1.1)
palette = sns.color_palette("Set2")

# ---------- 圖1：Accuracy ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_nb, x="資料處理方式", y="測試 Accuracy", hue="資料切分", palette=palette)
plt.title("Naive Bayes 各資料處理方式之測試集 Accuracy 比較")
plt.ylabel("Accuracy")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.65, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖2：F1 ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_nb, x="資料處理方式", y="F1", hue="資料切分", palette=palette)
plt.title("Naive Bayes 各資料處理方式之 F1 Score 比較")
plt.ylabel("F1 Score")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.6, 0.8)
plt.tight_layout()
plt.show()

# ---------- 圖3：AUC ----------
plt.figure(figsize=(8,5))
sns.barplot(data=df_nb, x="資料處理方式", y="AUC", hue="資料切分", palette=palette)
plt.title("Naive Bayes 各資料處理方式之 AUC 比較")
plt.ylabel("ROC AUC")
plt.xlabel("資料處理方式")
plt.legend(title="資料切分", loc="lower right")
plt.ylim(0.7, 0.85)
plt.tight_layout()
plt.show()


### ✅ SVM 模型

### 取SVM最好的結果來畫圖表

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
], remainder='passthrough')

split_size = len(df) // 3
results_std = []

for i in range(3):
    print(f"\n==== (標準化) 第 {i+1} 次資料分割與訓練 ====")
    start, end = i * split_size, (i + 1) * split_size if i < 2 else len(df)
    df_sub = df.iloc[start:end]
    X_sub, y_sub = df_sub.drop(columns=['cardio']), df_sub['cardio']

    X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42+i)

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(probability=True))
    ])

    param_grid = {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf'],
        'classifier__gamma': ['scale', 'auto']
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42+i)
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    print("最佳參數:", grid.best_params_)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"第{i+1}次結果：Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    results_std.append([acc, prec, rec, f1, auc])

df_std = pd.DataFrame(results_std, columns=['Accuracy','Precision','Recall','F1','AUC'])

print("\n=============================")
print("📊 SVM 三等份實驗結果")
print(df_std)
print("\n平均結果：\n", df_std.mean().to_frame().T)

## 🧩 步驟 6：模型結果分析與視覺化

### 從各模型 DataFrame 取最佳結果（依測試集 Accuracy 最大）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 從各模型 DataFrame 取最佳結果（依測試集 Accuracy 最大）
models = {
    "KNN": df_results,
    "Decision Tree": df_dt,
    "Random Forest": df_rf,
    "Naive Bayes": df_nb
}

summary = []

for name, df in models.items():
    best = df.loc[df["測試 Accuracy"].idxmax()]
    summary.append({
        "模型": name,
        "資料處理方式": best["資料處理方式"],
        "資料切分": best["資料切分"],
        "最佳參數": best["最佳參數"],
        "驗證平均準確率 (CV)": best["驗證平均準確率 (CV)"],
        "測試 Accuracy": best["測試 Accuracy"],
        "F1": best["F1"],
        "AUC": best["AUC"]
    })

# 加入 SVM 平均結果
svm_avg = df_std.mean()
summary.append({
    "模型": "SVM",
    "Accuracy": svm_avg["Accuracy"],
    "F1": svm_avg["F1"],
    "AUC": svm_avg["AUC"]
})

df_summary = pd.DataFrame(summary)
print("📊 五模型最佳表現比較表：")
display(df_summary)

### ROC Curve - 各模型比較

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# 模型設定
model_data = {
    "KNN": (best_knn_model, "blue"),
    "Decision Tree": (best_dt_model, "orange"),
    "Random Forest": (best_rf_model, "green"),
    "Naive Bayes": (best_nb_model, "purple"),
    "SVM": (best_model, "red")  # best_model 是你 SVM GridSearchCV 的最佳模型
}

plt.figure(figsize=(8,6))

for name, (model, color) in model_data.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], 'k--', label="Random Guess")
plt.title("ROC Curve - 五模型比較", fontsize=14, fontweight='bold')
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### 各模型最佳結果之混淆矩陣比較

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(12,8))
axes = axes.flatten()
model_list = list(model_data.items())

for i, (name, (model, color)) in enumerate(model_list):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[i])
    axes[i].set_title(f"{name}", fontsize=12, fontweight='bold')
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

# 留一格空白加上總標題
axes[-1].axis("off")
axes[-1].text(0.1, 0.5, "五模型最佳結果之混淆矩陣比較", fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
